### Determining the optimal number of hidden layers and neurons for an Artificial Neural Network (ANN) 
This can be challenging and often requires experimentation. However, there are some guidelines and methods that can help you in making an informed decision:

- Start Simple: Begin with a simple architecture and gradually increase complexity if needed.
- Grid Search/Random Search: Use grid search or random search to try different architectures.
- Cross-Validation: Use cross-validation to evaluate the performance of different architectures.
- Heuristics and Rules of Thumb: Some heuristics and empirical rules can provide starting points, such as:
  -    The number of neurons in the hidden layer should be between the size of the input layer and the size of the output layer.
  -  A common practice is to start with 1-2 hidden layers.

In [2]:
!pip install scikeras

  Using cached joblib-1.4.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached threadpoolctl-3.5.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/11.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.0 MB 2.0 MB/s eta 0:00:06
   ---------------------------------------- 0.1/11.0 MB 1.3 MB/s eta 0:00:09
    --------------------------------------- 0.1/11.0 MB 1.1 MB/s eta 0:00:11
    --------------------------------------- 0.2/11.0 MB 1.2 MB/s eta 0:00:09
   - -------------------------------------- 0.3/11.0 MB 1.3 MB/s eta 0:00:09
   - -------------------------------------- 0.4/11.0 MB 1.4 MB/s eta 0:00:08
   - -------------------------------------- 0.4/11.0 MB 1.4 MB/s eta 0:00:08
   - -------------------------------------- 0.5/11.0 MB 1.4 MB/s eta 0:00:08
   -- ------------------------------------- 0.6/11.0 MB 1.5 MB/s eta 0:00:08
   -- ------------------------------------- 0.7/11.0 MB 1.5 MB/s eta 0:00:07
   -- ----------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pandas-profiling 3.2.0 requires joblib~=1.1.0, but you have joblib 1.4.2 which is incompatible.


In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import pickle

In [4]:
data=pd.read_csv('Churn_Modelling.csv')
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

onehot_encoder_geo = OneHotEncoder(handle_unknown='ignore')
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)

X = data.drop('Exited', axis=1)
y = data['Exited']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Save encoders and scaler for later use
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)

with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

In [5]:
#function to create a model and try different parameters(keras classifier)
def create_model(neurons=32,layers=1):
    model = Sequential()
    model.add(Dense(neurons,activation='relu',input_shape=(X_train.shape[1],)))
    for _ in range(layers-1):
        model.add(Dense(neurons,activation ='relu'))

    model.add(Dense(1,activation='sigmoid'))
    model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
    return model

In [6]:
## Create a Keras classifier
model=KerasClassifier(layers=1,neurons=32,build_fn=create_model,verbose=1)

In [7]:

# Define the grid search parameters
param_grid = {
    'neurons': [16, 32, 64, 128],
    'layers': [1, 2],
    'epochs': [50, 100]
}

In [9]:
# Perform grid search
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=1, cv=3,verbose=1)
grid_result = grid.fit(X_train, y_train)

# Print the best parameters
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))

Fitting 3 folds for each of 16 candidates, totalling 48 fits
Epoch 1/50


c:\Users\kumma\anaconda3\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
c:\Users\kumma\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 924us/step - accuracy: 0.6866 - loss: 0.6067
Epoch 2/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 646us/step - accuracy: 0.8007 - loss: 0.4692
Epoch 3/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 620us/step - accuracy: 0.8081 - loss: 0.4376
Epoch 4/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 625us/step - accuracy: 0.8169 - loss: 0.4177
Epoch 5/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 601us/step - accuracy: 0.8157 - loss: 0.4159
Epoch 6/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 640us/step - accuracy: 0.8167 - loss: 0.4181
Epoch 7/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 655us/step - accuracy: 0.8296 - loss: 0.3996
Epoch 8/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 603us/step - accuracy: 0.8381 - loss: 0.3844
Epoch 9/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 631us/step - accuracy: 0.8406 - loss: 0.3894
Epoch 10/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 664us/step - accuracy: 0.8474 - loss: 0.3758
Epoch 11/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 732us/step - accuracy: 0.8546 - loss: 0.3556
Epoch 12/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 

c:\Users\kumma\anaconda3\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
c:\Users\kumma\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 654us/step - accuracy: 0.5534 - loss: 0.7645
Epoch 2/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 709us/step - accuracy: 0.8062 - loss: 0.4851
Epoch 3/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 637us/step - accuracy: 0.8048 - loss: 0.4512
Epoch 4/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 698us/step - accuracy: 0.8153 - loss: 0.4250
Epoch 5/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 653us/step - accuracy: 0.8185 - loss: 0.4248
Epoch 6/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 703us/step - accuracy: 0.8299 - loss: 0.4065
Epoch 7/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 604us/step - accuracy: 0.8356 - loss: 0.3990
Epoch 8/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 682us/step - accuracy: 0.8440 - loss: 0.3906
Epoch 9/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 630us/step - accuracy: 0.8463 - loss: 0.3796
Epoch 10/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 674us/step - accuracy: 0.8542 - loss: 0.3718
Epoch 11/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 828us/step - accuracy: 0.8558 - loss: 0.3601
Epoch 12/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 

...

167/167 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.7900 - loss: 0.4765
Epoch 2/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 799us/step - accuracy: 0.8543 - loss: 0.3667
Epoch 3/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 845us/step - accuracy: 0.8638 - loss: 0.3474
Epoch 4/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 792us/step - accuracy: 0.8635 - loss: 0.3358
Epoch 5/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 810us/step - accuracy: 0.8695 - loss: 0.3342
Epoch 6/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 797us/step - accuracy: 0.8625 - loss: 0.3314
Epoch 7/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 798us/step - accuracy: 0.8656 - loss: 0.3213
Epoch 8/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 801us/step - accuracy: 0.8673 - loss: 0.3231
Epoch 9/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 810us/step - accuracy: 0.8739 - loss: 0.3084
Epoch 10/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 812us/step - accuracy: 0.8765 - loss: 0.3054
Epoch 11/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 825us/step - accuracy: 0.8714 - loss: 0.3091
Epoch 12/100
167/167 ━━━━━━━━━━━━

c:\Users\kumma\anaconda3\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
c:\Users\kumma\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


167/167 ━━━━━━━━━━━━━━━━━━━━ 2s 841us/step - accuracy: 0.7695 - loss: 0.4884
Epoch 2/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 871us/step - accuracy: 0.8514 - loss: 0.3721
Epoch 3/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 822us/step - accuracy: 0.8513 - loss: 0.3535
Epoch 4/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 823us/step - accuracy: 0.8494 - loss: 0.3519
Epoch 5/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 815us/step - accuracy: 0.8707 - loss: 0.3187
Epoch 6/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 809us/step - accuracy: 0.8636 - loss: 0.3237
Epoch 7/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 807us/step - accuracy: 0.8697 - loss: 0.3146
Epoch 8/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 821us/step - accuracy: 0.8709 - loss: 0.3154
Epoch 9/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 808us/step - accuracy: 0.8643 - loss: 0.3192
Epoch 10/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 808us/step - accuracy: 0.8802 - loss: 0.2958
Epoch 11/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 882us/step - accuracy: 0.8765 - loss: 0.3017
Epoch 12/100
167/167 ━━━━━━━━━━

c:\Users\kumma\anaconda3\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
c:\Users\kumma\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 649us/step - accuracy: 0.6324 - loss: 0.6342
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 641us/step - accuracy: 0.8110 - loss: 0.4386
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 638us/step - accuracy: 0.8271 - loss: 0.4117
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 627us/step - accuracy: 0.8358 - loss: 0.3935
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 647us/step - accuracy: 0.8487 - loss: 0.3736
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 629us/step - accuracy: 0.8557 - loss: 0.3597
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 636us/step - accuracy: 0.8580 - loss: 0.3568
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 649us/step - accuracy: 0.8576 - loss: 0.3575
Epoch 9/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 638us/step - accuracy: 0.8680 - loss: 0.3428
Epoch 10/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 761us/step - accuracy: 0.8601 - loss: 0.3472
Epoch 11/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 640us/step - accuracy: 0.8595 - loss: 0.3440
Epoch 12/100
250/250 ━━━━━━━━━━